# Topstep 50K Fast Pipeline

Run this notebook top-to-bottom. Set `run_mode` to `FAST` for quick iteration or `FULL` for full training.


In [1]:
# Setup
import os
import sys
import json as pyjson
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, JSON

os.environ.setdefault("RISK_PRESET_NAME", "TOPSTEP_50K")
os.environ.setdefault("ES_BARS_H5", "data/processed/es_bars_2010_2025.h5")

run_mode = "FAST"  # FAST or FULL
data_path = os.environ["ES_BARS_H5"]
dataset_key = "bars_5min"
model_dir = "models/nn_saved"
fold = 0

fast_max_bars = 250_000

project_root = Path().resolve().parent if Path().resolve().name == 'analysis' else Path().resolve()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from core.risk_presets import get_risk_preset
from core.selection import bars_per_day
from data.clean_bars import clean_bars
from features.labels_aligned import make_aligned_fixed_horizon_labels
from models.nn_inference import (
    load_nn_bundle,
    predict_scores_for_bars,
    artifact_compatibility_issues,
)
from backtesting.backtest import run_backtest_nn
from analysis.monte_carlo_combine import simulate_combine


In [2]:
# Load and clean bars
if not Path(data_path).exists():
    raise FileNotFoundError(f"Missing data file: {data_path}")

with pd.HDFStore(data_path, "r") as store:
    if dataset_key not in store:
        raise KeyError(f"Dataset key {dataset_key!r} not found in H5.")
    bars = store[dataset_key].copy()

bars["timestamp"] = pd.to_datetime(bars["timestamp"], utc=True)
bars = bars.sort_values("timestamp").reset_index(drop=True)

preset = get_risk_preset("TOPSTEP_50K")
bars = clean_bars(bars, tick_size=preset.risk_config.tick_size, verbose=True)
if run_mode.upper() == "FAST":
    bars = bars.tail(fast_max_bars).reset_index(drop=True)
assert bars["timestamp"].is_monotonic_increasing, "Bars must be time-ordered"

print(f"Loaded {len(bars):,} bars")
print(f"Date range: {bars['timestamp'].min()} to {bars['timestamp'].max()}")



BAR CLEANING REPORT
Input rows:  306,933
Output rows: 306,933  (removed 0)

Loaded 250,000 bars
Date range: 2013-04-26 16:35:00+00:00 to 2025-12-19 20:55:00+00:00


In [3]:
# Train or reuse NN artifacts
config_path = Path(model_dir) / f"fold_{fold}" / "config.json"

def needs_retrain(path: Path) -> bool:
    if not path.exists():
        return True
    try:
        cfg = pyjson.loads(path.read_text())
    except Exception:
        return True
    issues = artifact_compatibility_issues(cfg, strict_versions=True)
    if issues:
        print(f"Artifact incompatibility: {issues}")
        return True
    return False

if needs_retrain(config_path):
    train_cmd = [
        sys.executable,
        "models/nn_train.py",
        "--data-path",
        data_path,
        "--dataset-key",
        dataset_key,
        "--output-dir",
        model_dir,
    ]
    if run_mode.upper() == "FAST":
        train_cmd += ["--fast", "--max-bars", str(fast_max_bars)]
    print('Training command:', ' '.join(train_cmd))
    subprocess.run(train_cmd, check=True)
else:
    print(f"Using existing model artifacts at {config_path}")


Using existing model artifacts at models/nn_saved/fold_0/config.json


In [4]:
# Load model bundle + label diagnostics
bundle = load_nn_bundle(model_dir, fold=fold)
nn_cfg = bundle.config.get("nn_config", {})

# Print resolved dynamic stop configuration
print("=" * 60)
print("RESOLVED CONFIGURATION")
print("=" * 60)
print(f"Artifact path: {model_dir}/fold_{fold}")
print(f"\nDynamic Stop Configuration:")
print(f"  use_dynamic_catastop: {nn_cfg.get('use_dynamic_catastop', False)}")
print(f"  catastop_atr_multiplier: {nn_cfg.get('catastop_atr_multiplier', 3.5):.1f}x")
print(f"  catastop_min_ticks: {nn_cfg.get('catastop_min_ticks', 24)}")
print(f"  catastop_max_ticks: {nn_cfg.get('catastop_max_ticks', 140)}")
print(f"\nPosition Sizing:")
print(f"  risk_per_trade_usd: ${nn_cfg.get('risk_per_trade_usd', 200):.0f}")
print(f"  max_contracts: {nn_cfg.get('max_contracts', 5)}")
print(f"\nSelection:")
print(f"  selection_mode: {nn_cfg.get('selection_mode', 'global_threshold')}")
print(f"  score_threshold: {nn_cfg.get('score_threshold', 0.5):.4f}")
print(f"  max_trades_per_day: {nn_cfg.get('max_trades_per_day', 2)}")
print("=" * 60)

nn_cfg_display = {k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in nn_cfg.items()}
print("\nFull NN artifact config:")
display(JSON(nn_cfg_display, indent=2))

labels_df = make_aligned_fixed_horizon_labels(
    bars,
    horizon_bars=int(nn_cfg["horizon_bars"]),
    threshold_ticks=int(nn_cfg["threshold_ticks"]),
    tick_size=float(nn_cfg["tick_size"]),
    entry_price_col=str(nn_cfg.get("label_entry_price_col", "open")),
    exit_price_col=str(nn_cfg.get("label_exit_price_col", "close")),
)
print(f"\nAvg |ret_ticks|: {labels_df['ret_ticks'].abs().mean():.2f}")

prob_df = predict_scores_for_bars(bars, bundle)
scores = prob_df["score"].dropna()
if not scores.empty:
    percentiles = {f"p{p}": float(np.nanpercentile(scores, p)) for p in [50, 90, 95, 97, 98, 99, 99.5]}
else:
    percentiles = {}
print("\nScore percentiles:")
print(pyjson.dumps(percentiles, indent=2))


RESOLVED CONFIGURATION
Artifact path: models/nn_saved/fold_0

Dynamic Stop Configuration:
  use_dynamic_catastop: True
  catastop_atr_multiplier: 3.5x
  catastop_min_ticks: 24
  catastop_max_ticks: 140

Position Sizing:
  risk_per_trade_usd: $200
  max_contracts: 5

Selection:
  selection_mode: day_adaptive_topn
  score_threshold: 0.4611
  max_trades_per_day: 4

Full NN artifact config:


<IPython.core.display.JSON object>


Avg |ret_ticks|: 33.28

Score percentiles:
{
  "p50": 0.40742279048655605,
  "p90": 0.45705218551960025,
  "p95": 0.474061131074805,
  "p97": 0.4889555820103318,
  "p98": 0.5024061351693494,
  "p99": 0.5310871384890243,
  "p99.5": 0.5711316966752913
}


In [5]:
# Backtest (ML-only, aligned time-exit with dynamic catastrophic stops)
risk_cfg = preset.risk_config
results = run_backtest_nn(
    bars,
    prob_df,
    score_threshold=float(nn_cfg["score_threshold"]),
    selection_mode=str(nn_cfg.get("selection_mode", "global_threshold")),
    day_percentile_floor=float(nn_cfg.get("day_percentile_floor", 0.90)),
    global_floor_score=float(nn_cfg.get("global_floor_score", nn_cfg["score_threshold"])),
    max_trades_per_day=int(nn_cfg["max_trades_per_day"]),
    min_bars_between_trades=int(nn_cfg["min_bars_between_trades"]),
    enable_long=bool(nn_cfg["enable_long"]),
    enable_short=bool(nn_cfg["enable_short"]),
    horizon_bars=int(nn_cfg["horizon_bars"]),
    execution_mode=str(nn_cfg["execution_mode"]),
    exit_price_mode=str(nn_cfg["exit_price_mode"]),
    session_mode=str(nn_cfg["session_mode"]),
    deadline_time=nn_cfg.get("deadline_time"),
    deadline_relax_factor=float(nn_cfg.get("deadline_relax_factor", 0.98)),
    bar_minutes=int(nn_cfg["bar_minutes"]),
    session_start=risk_cfg.session_start,
    session_end=risk_cfg.session_end,
    stop_loss_ticks=int(nn_cfg["stop_loss_ticks"]),
    target_multiplier=float(nn_cfg["target_multiplier"]),
    catastrophic_stop_ticks=int(nn_cfg.get("catastrophic_stop_ticks", int(nn_cfg["threshold_ticks"]) * 4)),
    max_hold_bars=int(nn_cfg["max_hold_bars"]),
    tick_size=float(nn_cfg["tick_size"]),
    tick_value=float(nn_cfg["tick_value"]),
    # Dynamic catastrophic stop parameters (from artifact config)
    use_dynamic_catastop=bool(nn_cfg.get("use_dynamic_catastop", True)),
    catastop_atr_multiplier=float(nn_cfg.get("catastop_atr_multiplier", 3.5)),
    catastop_min_ticks=int(nn_cfg.get("catastop_min_ticks", 24)),
    catastop_max_ticks=int(nn_cfg.get("catastop_max_ticks", 72)),
    save_trades_path="analysis/notebook_backtest_trades_50k.csv",
)

print("=" * 60)
print("BACKTEST RESULTS")
print("=" * 60)

print("\nBacktest summary:")
print(pyjson.dumps(results["summary"], indent=2))

print("\nDaily stats:")
print(pyjson.dumps(results["daily_stats"], indent=2))

print("\nExit reasons:")
exit_reasons = results.get("exit_reason_counts", {})
print(pyjson.dumps(exit_reasons, indent=2))

print("\nExit reason avg PnL:")
print(pyjson.dumps(results.get("exit_reason_avg_pnl", {}), indent=2))

# CATASTOP rate (prominent)
print("\n" + "=" * 60)
catastop_rate = results.get("catastop_rate", 0)
print(f"⚠️  CATASTOP RATE: {catastop_rate:.2f}%")
print("=" * 60)

# Dynamic stop diagnostics
if "stop_diagnostics" in results and results["stop_diagnostics"]:
    print("\n" + "=" * 60)
    print("DYNAMIC STOP DIAGNOSTICS")
    print("=" * 60)
    
    diag = results["stop_diagnostics"]
    
    if "stop_ticks_distribution" in diag:
        print("\nStop ticks distribution:")
        print(pyjson.dumps(diag["stop_ticks_distribution"], indent=2))
    
    if "atr_ticks_distribution" in diag:
        print("\nATR ticks distribution (at entry):")
        print(pyjson.dumps(diag["atr_ticks_distribution"], indent=2))
    
    if "stop_atr_ratio_distribution" in diag:
        print("\nStop/ATR ratio distribution:")
        print(pyjson.dumps(diag["stop_atr_ratio_distribution"], indent=2))
    
    if "atr_fallback_count" in diag:
        print(f"\nATR fallback count (missing ATR): {diag['atr_fallback_count']}")

bars_day = bars_per_day(
    session_mode=str(nn_cfg.get("session_mode", "RTH")),
    session_start=risk_cfg.session_start,
    session_end=risk_cfg.session_end,
    bar_minutes=int(nn_cfg["bar_minutes"]),
)
print(f"\nBars/day (session): {bars_day} | Target trades/day: {nn_cfg.get('target_trades_per_day')}")

trades = pd.read_csv("analysis/notebook_backtest_trades_50k.csv")
assert len(trades) > 0, "Backtest produced 0 trades; aborting."


BACKTEST RESULTS

Backtest summary:
{
  "trades": 34,
  "wins": 10,
  "losses": 24,
  "win_rate": 0.29411764705882354,
  "profit_factor": 0.3179506380854317,
  "gross_wins": 1019.0,
  "gross_losses": 3204.8999999999996,
  "net_pnl": -2185.8999999999996,
  "avg_pnl": -64.29117647058823,
  "max_drawdown": 2185.899999999994,
  "ending_equity": 47814.100000000006,
  "starting_balance": 50000.0
}

Daily stats:
{
  "calendar_days": 3254,
  "rth_days_in_data": 3250,
  "rth_days_with_trades": 31,
  "pct_rth_days_traded": 0.9538461538461539,
  "total_trading_days": 3254,
  "avg_trades_per_day": 0.010448678549477565,
  "avg_trades_per_active_day": 1.096774193548387,
  "max_trades_in_day": 3,
  "days_with_trades": 31,
  "days_with_zero_trades": 3223,
  "trades_per_day_distribution": {
    "0": 3223,
    "1": 29,
    "2": 1,
    "3": 1
  },
  "pct_days_with_1_trade": 0.8912108174554395,
  "pct_days_with_2_trades": 0.030731407498463426
}

Exit reasons:
{
  "TIME_EXIT": 25,
  "CATASTOP": 9
}

Exit r

In [6]:
# Trades diagnostics
trades_df = pd.DataFrame(results.get("trades", []))
if trades_df.empty:
    print("No trades produced.")
else:
    trades_df["entry_time"] = pd.to_datetime(trades_df["entry_time"], utc=True)
    trades_df["day"] = trades_df["entry_time"].dt.date
    trades_per_day = trades_df.groupby("day")["pnl"].count()
    print(f"Avg trades/day: {trades_per_day.mean():.2f}")
    print(f"Win rate: {(trades_df['pnl'] > 0).mean():.2%}")
    print(f"Avg pnl/trade: {trades_df['pnl'].mean():.2f}")
    wins = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
    losses = trades_df[trades_df['pnl'] <= 0]['pnl'].sum()
    pf = wins / abs(losses) if losses != 0 else float('inf')
    print(f"Profit factor: {pf:.2f}")


Avg trades/day: 1.10
Win rate: 29.41%
Avg pnl/trade: -64.29
Profit factor: 0.32


In [7]:
# CATASTOP Analysis
print("=" * 60)
print("CATASTROPHIC STOP ANALYSIS")
print("=" * 60)

if "reason" in trades_df.columns:
    catastop_trades = trades_df[trades_df["reason"] == "CATASTOP"]
    time_exit_trades = trades_df[trades_df["reason"] == "TIME_EXIT"]
    
    total = len(trades_df)
    catastop_count = len(catastop_trades)
    catastop_rate = (catastop_count / total * 100) if total > 0 else 0
    
    print(f"\nCATASTOP trades: {catastop_count}/{total} ({catastop_rate:.2f}%)")
    print(f"TIME_EXIT trades: {len(time_exit_trades)}/{total} ({len(time_exit_trades)/total*100:.2f}%)")
    
    if catastop_count > 0:
        print(f"\nCATASTOP avg PnL: ${catastop_trades['pnl'].mean():.2f}")
        print(f"CATASTOP total loss: ${catastop_trades['pnl'].sum():.2f}")
        
    if len(time_exit_trades) > 0:
        print(f"\nTIME_EXIT avg PnL: ${time_exit_trades['pnl'].mean():.2f}")
        print(f"TIME_EXIT total: ${time_exit_trades['pnl'].sum():.2f}")
        print(f"TIME_EXIT win rate: {(time_exit_trades['pnl'] > 0).mean():.2%}")
    
    # Show impact
    if catastop_count > 0 and len(time_exit_trades) > 0:
        catastop_drag = catastop_trades['pnl'].sum()
        time_exit_profit = time_exit_trades['pnl'].sum()
        print(f"\n💡 Impact: CATASTOP losses ({catastop_drag:.2f}) offset {abs(catastop_drag/time_exit_profit)*100:.1f}% of TIME_EXIT gains")


CATASTROPHIC STOP ANALYSIS

CATASTOP trades: 9/34 (26.47%)
TIME_EXIT trades: 25/34 (73.53%)

CATASTOP avg PnL: $-235.38
CATASTOP total loss: $-2118.45

TIME_EXIT avg PnL: $-2.70
TIME_EXIT total: $-67.45
TIME_EXIT win rate: 40.00%

💡 Impact: CATASTOP losses (-2118.45) offset 3140.8% of TIME_EXIT gains


In [8]:
# Monte Carlo combine pass-rate
if trades_df.empty:
    raise ValueError("No trades available for Monte Carlo simulation.")
combine = simulate_combine(
    trades_df,
    starting_balance=preset.risk_config.starting_balance,
    profit_target=preset.profit_target,
    daily_loss_limit=preset.risk_config.max_daily_loss,
    trailing_drawdown=preset.risk_config.trailing_drawdown,
    runs=5000 if run_mode.upper() == "FAST" else 20000,
    seed=42,
    max_days=252,
    consistency_limit=preset.consistency_limit,
)

print("Topstep 50K pass-rate summary:")
print(pyjson.dumps(combine, indent=2))
if combine.get('pass_rate', 0) > 0 and combine.get('days_to_pass'):
    print(f"Median days to pass: {combine['days_to_pass'].get('p50')}")


Topstep 50K pass-rate summary:
{
  "runs": 5000,
  "pass_rate": 0.0,
  "days_to_pass": {},
  "fail_reasons": {
    "daily_loss": 0,
    "trailing_drawdown": 5000,
    "consistency_limit": 0,
    "max_days": 0
  },
  "max_drawdown": {
    "p05": 2006.1974999999838,
    "p50": 2079.0499999999956,
    "p95": 2243.454999999983,
    "mean": 2095.9573799999953
  }
}


In [9]:
# Final Summary Report
print("\n" + "=" * 60)
print("FINAL SUMMARY REPORT")
print("=" * 60)

print(f"\nModel: {model_dir}/fold_{fold}")
print(f"Data: {data_path} ({len(bars):,} bars)")
print(f"Run mode: {run_mode}")

print("\n" + "-" * 60)
print("CONFIGURATION")
print("-" * 60)
print(f"Dynamic stops: {nn_cfg.get('use_dynamic_catastop', False)}")
print(f"ATR Multiplier: {nn_cfg.get('catastop_atr_multiplier', 3.5):.1f}x")
print(f"Stop range: [{nn_cfg.get('catastop_min_ticks', 24)}, {nn_cfg.get('catastop_max_ticks', 140)}] ticks")
print(f"Risk/trade: ${nn_cfg.get('risk_per_trade_usd', 200):.0f}")
print(f"Max contracts: {nn_cfg.get('max_contracts', 5)}")

print("\n" + "-" * 60)
print("BACKTEST PERFORMANCE")
print("-" * 60)
summary = results["summary"]
print(f"Trades: {summary['trades']}")
print(f"Win Rate: {summary['win_rate']:.2%}")
print(f"Profit Factor: {summary['profit_factor']:.2f}")
print(f"Net PnL: ${summary['net_pnl']:.2f}")
print(f"Avg PnL: ${summary['avg_pnl']:.2f}")
print(f"Max Drawdown: ${summary['max_drawdown']:.2f}")

print("\n" + "-" * 60)
print("CATASTOP ANALYSIS")
print("-" * 60)
catastop_rate = results.get('catastop_rate', 0)
print(f"⚠️  CATASTOP Rate: {catastop_rate:.2f}%")
exit_reasons = results.get("exit_reason_counts", {})
catastop_count = exit_reasons.get("CATASTOP", 0)
time_exit_count = exit_reasons.get("TIME_EXIT", 0)
print(f"CATASTOP: {catastop_count}/{summary['trades']} trades")
print(f"TIME_EXIT: {time_exit_count}/{summary['trades']} trades")

# Position sizing diagnostics
if "contract_distribution" in results and results["contract_distribution"]:
    cd = results["contract_distribution"]
    print(f"\nContracts: p50={cd['p50']:.1f}, p90={cd['p90']:.1f}, mean={cd['mean']:.2f}")
if "dollar_risk_distribution" in results and results["dollar_risk_distribution"]:
    dr = results["dollar_risk_distribution"]
    print(f"Dollar risk: p50=${dr['p50']:.0f}, p90=${dr['p90']:.0f}, target=${dr['target']:.0f}")

# Stop diagnostics
if "stop_diagnostics" in results and results["stop_diagnostics"]:
    diag = results["stop_diagnostics"]
    if "stop_ticks_distribution" in diag:
        stop_dist = diag["stop_ticks_distribution"]
        print(f"\nStop ticks: p50={stop_dist['p50']:.0f}, p90={stop_dist['p90']:.0f}, mean={stop_dist['mean']:.1f}")
    if "atr_ticks_distribution" in diag:
        atr_dist = diag["atr_ticks_distribution"]
        print(f"ATR ticks:  p50={atr_dist['p50']:.1f}, p90={atr_dist['p90']:.1f}, mean={atr_dist['mean']:.1f}")
    if "stop_atr_ratio_distribution" in diag:
        ratio_dist = diag["stop_atr_ratio_distribution"]
        print(f"Stop/ATR:   p50={ratio_dist['p50']:.2f}x, p90={ratio_dist['p90']:.2f}x, mean={ratio_dist['mean']:.2f}x")

# V2 backtest diagnostics
if "stop_ticks_distribution" in results and results["stop_ticks_distribution"]:
    std = results["stop_ticks_distribution"]
    print(f"\nStop ticks: p50={std['p50']:.0f}, p90={std['p90']:.0f}, mean={std['mean']:.1f}")

print("\n" + "-" * 60)
print("TOPSTEP 50K MONTE CARLO")
print("-" * 60)
print(f"Pass Rate: {combine.get('pass_rate', 0)*100:.2f}%")
if combine.get('pass_rate', 0) > 0:
    print(f"Median Days to Pass: {combine['days_to_pass'].get('p50', 0):.0f}")
print(f"Fail (Trailing DD): {combine['fail_reasons'].get('trailing_drawdown', 0):,}")
print(f"Fail (Max Days): {combine['fail_reasons'].get('max_days', 0):,}")

# Goal check
goal_met = catastop_rate < 15 and summary['profit_factor'] > 1.0
status = "✅ GOAL MET" if goal_met else "⚠️  NEEDS IMPROVEMENT"
print(f"\n{status}: CATASTOP < 15% and PF > 1.0")
print("=" * 60)



FINAL SUMMARY REPORT

Model: models/nn_saved/fold_0
Data: data/processed/es_bars_2010_2025.h5 (250,000 bars)
Run mode: FAST

------------------------------------------------------------
CONFIGURATION
------------------------------------------------------------
Dynamic stops: True
ATR Multiplier: 3.5x
Stop range: [24, 140] ticks
Risk/trade: $200
Max contracts: 5

------------------------------------------------------------
BACKTEST PERFORMANCE
------------------------------------------------------------
Trades: 34
Win Rate: 29.41%
Profit Factor: 0.32
Net PnL: $-2185.90
Avg PnL: $-64.29
Max Drawdown: $2185.90

------------------------------------------------------------
CATASTOP ANALYSIS
------------------------------------------------------------
⚠️  CATASTOP Rate: 26.47%
CATASTOP: 9/34 trades
TIME_EXIT: 25/34 trades

Contracts: p50=3.0, p90=4.0, mean=2.85
Dollar risk: p50=$161, p90=$194, target=$200

Stop ticks: p50=44, p90=73, mean=52.0
ATR ticks:  p50=12.9, p90=21.1, mean=16.3
Stop/